<a href="https://colab.research.google.com/github/njalagam23/neural-networks-from-scratch/blob/main/day10_fastapi_churn_prediction_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!git clone https://github.com/njalagam23/neural-networks-from-scratch.git /content/churn-repo

fatal: destination path '/content/churn-repo' already exists and is not an empty directory.


In [32]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

required_files = [
    "app/model.py",
    "app/preprocessing.py",
    "app/predict.py",
    "artifacts/churn_model.pth",
    "artifacts/churn_scaler.joblib",
    "artifacts/model_config.json"
]

for filename in required_files:
    print(filename, ":", (project / filename).exists())

app/model.py : True
app/preprocessing.py : True
app/predict.py : True
artifacts/churn_model.pth : True
artifacts/churn_scaler.joblib : True
artifacts/model_config.json : True


In [33]:
!pip -q install fastapi uvicorn httpx

In [34]:
%%writefile /content/churn-repo/customer-churn-app/api.py

from fastapi import FastAPI
from pydantic import BaseModel, Field

from app.predict import ChurnPredictor


# Create the FastAPI application
app = FastAPI(
    title="Customer Churn Prediction API",
    version="1.0.0"
)

# Load our trained model once
predictor = ChurnPredictor()


# Define the expected customer input
class CustomerInput(BaseModel):
    usage: float = Field(ge=0)
    tenure: int = Field(ge=0)
    tickets: int = Field(ge=0)


# Home endpoint
@app.get("/")
def home():
    return {"message": "Churn Prediction API is running"}


# Prediction endpoint
@app.post("/predict")
def predict_churn(customer: CustomerInput):

    result = predictor.predict(
        usage=customer.usage,
        tenure=customer.tenure,
        tickets=customer.tickets
    )

    return result

Overwriting /content/churn-repo/customer-churn-app/api.py


In [35]:
import sys

sys.path.insert(
    0,
    "/content/churn-repo/customer-churn-app"
)

from fastapi.testclient import TestClient
from api import app

client = TestClient(app)

response = client.get("/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'Churn Prediction API is running'}


In [36]:
response = client.post(
    "/predict",
    json={
        "usage": 5,
        "tenure": 8,
        "tickets": 3
    }
)

print("Status code:", response.status_code)
print("Prediction:", response.json())

Status code: 200
Prediction: {'churn_probability': 0.7995, 'risk_level': 'High Churn Risk'}


In [37]:
invalid_customers = [
    {
        "usage": -5,
        "tenure": 8,
        "tickets": 3
    },
    {
        "usage": 5,
        "tickets": 3
    },
    {
        "usage": "unknown",
        "tenure": 8,
        "tickets": 3
    },
    {
        "usage": 5,
        "tenure": 8,
        "tickets": -1
    }
]

for customer in invalid_customers:
    response = client.post(
        "/predict",
        json=customer
    )

    print("Customer:", customer)
    print("Status code:", response.status_code)
    print("-" * 40)

Customer: {'usage': -5, 'tenure': 8, 'tickets': 3}
Status code: 422
----------------------------------------
Customer: {'usage': 5, 'tickets': 3}
Status code: 422
----------------------------------------
Customer: {'usage': 'unknown', 'tenure': 8, 'tickets': 3}
Status code: 422
----------------------------------------
Customer: {'usage': 5, 'tenure': 8, 'tickets': -1}
Status code: 422
----------------------------------------


In [38]:
response = client.post(
    "/predict",
    json={
        "usage": -5,
        "tenure": 8,
        "tickets": 3
    }
)

print("Status code:", response.status_code)
print("Error details:", response.json()["detail"])

Status code: 422
Error details: [{'type': 'greater_than_equal', 'loc': ['body', 'usage'], 'msg': 'Input should be greater than or equal to 0', 'input': -5, 'ctx': {'ge': 0.0}}]


In [39]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

print("Project exists:", project.exists())
print("Tests folder exists:", (project / "tests").exists())
print("API file exists:", (project / "api.py").exists())

Project exists: True
Tests folder exists: True
API file exists: True


In [40]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

tests_folder = project / "tests"
tests_folder.mkdir(parents=True, exist_ok=True)

print("Tests folder exists:", tests_folder.exists())

Tests folder exists: True


In [41]:
%%writefile /content/churn-repo/customer-churn-app/tests/test_api.py

import pytest
from fastapi.testclient import TestClient
from api import app

client = TestClient(app)


def test_home_endpoint():
    response = client.get("/")
    assert response.status_code == 200


def test_valid_prediction():
    response = client.post(
        "/predict",
        json={"usage": 5, "tenure": 8, "tickets": 3}
    )

    assert response.status_code == 200
    assert 0 <= response.json()["churn_probability"] <= 1


@pytest.mark.parametrize("customer", [
    {"usage": -5, "tenure": 8, "tickets": 3},
    {"usage": 5, "tickets": 3},
    {"usage": "unknown", "tenure": 8, "tickets": 3},
    {"usage": 5, "tenure": 8, "tickets": -1}
])
def test_invalid_requests(customer):
    response = client.post("/predict", json=customer)
    assert response.status_code == 422

Overwriting /content/churn-repo/customer-churn-app/tests/test_api.py


In [42]:
!cd /content/churn-repo/customer-churn-app && python -m pytest tests/test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 6 items                                                              

tests/test_api.py::test_home_endpoint PASSED                             [ 16%]
tests/test_api.py::test_valid_prediction PASSED                          [ 33%]
tests/test_api.py::test_invalid_requests[customer0] PASSED               [ 50%]
tests/test_api.py::test_invalid_requests[customer1] PASSED               [ 66%]
tests/test_api.py::test_invalid_requests[customer2] PASSED               [ 83%]
tests/test_api.py::test_invalid_requests[customer3] PASSED               [100%]

============================== 6 passed in 11.82s ==============================


In [43]:
!cd /content/churn-repo && git pull

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 15 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 8.23 KiB | 1.18 MiB/s, done.
From https://github.com/njalagam23/neural-networks-from-scratch
   ea327e2..f3de37b  main       -> origin/main
Updating ea327e2..f3de37b
Fast-forward
 customer-churn-app/Tests/test_api.py               |  32 ++
 .../Tests}/test_prediction.py                      |   0
 day10_fastapi_churn_prediction_ipynb.ipynb         | 527 +++++++++++++++++++++
 3 files changed, 559 insertions(+)
 create mode 100644 customer-churn-app/Tests/test_api.py
 rename {Tests => customer-churn-app/Tests}/test_prediction.py (100%)
 create mode 100644 day10_fastapi_churn_prediction_ipynb.ipynb


In [44]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

print("Tests folder:", (project / "tests").exists())
print("Day 9 tests:", (project / "tests/test_prediction.py").exists())
print("Day 10 API:", (project / "api.py").exists())

Tests folder: True
Day 9 tests: True
Day 10 API: True


In [45]:
from pathlib import Path

tests_folder = Path(
    "/content/churn-repo/customer-churn-app/tests"
)

for file in tests_folder.iterdir():
    print(file.name)

test_prediction.py
test_api.py
__pycache__


In [46]:
from pathlib import Path

repo = Path("/content/churn-repo")

for file in repo.rglob("*.py"):
    if "test" in file.name.lower() and "__pycache__" not in str(file):
        print(file.relative_to(repo))

customer-churn-app/tests/test_prediction.py
customer-churn-app/tests/test_api.py
customer-churn-app/Tests/test_prediction.py
customer-churn-app/Tests/test_api.py


In [48]:
tests_folder = repo / "customer-churn-app/tests"

for file in tests_folder.glob("test_*.py"):
    print(file.name)

test_prediction.py
test_api.py


In [49]:
!cd /content/churn-repo/customer-churn-app && python -m pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 16 items                                                             

tests/test_api.py::test_home_endpoint PASSED                             [  6%]
tests/test_api.py::test_valid_prediction PASSED                          [ 12%]
tests/test_api.py::test_invalid_requests[customer0] PASSED               [ 18%]
tests/test_api.py::test_invalid_requests[customer1] PASSED               [ 25%]
tests/test_api.py::test_invalid_requests[customer2] PASSED               [ 31%]
tests/test_api.py::test_invalid_requests[customer3] PASSED               [ 37%]
tests/test_prediction.py::test_prediction_probability PASSED             [ 43%]
tests/test_prediction.py::test_customer_risk[5-